# NFCorpus + Contextual Retrieval (chunk-then-contextualize)

**Context-Enriched Retrieval for RAG — Iris.ai**

Anthropic's **Contextual Retrieval**: before embedding a chunk, an LLM writes a short blurb that
situates the chunk **within its parent document** (resolving references, adding section/topic context).
That context is prepended to the chunk before embedding, so a fragment that is meaningless alone
becomes self-contained.

**Adapting to NFCorpus:** its corpus entries are whole abstracts, so we first **chunk** each abstract,
then treat the full abstract as the "document" when generating each chunk's context. We retrieve
chunks and **pool back to the parent doc** (qrels are at doc level).

**Built-in ablation (to attribute the gain):**
1. `baseline` — whole-doc embeddings (from the baseline notebook)
2. `chunks-only` — chunked, **no** context (isolates the effect of chunking)
3. `contextual` — chunked **+** LLM context (Contextual Retrieval)

Comparing 2→3 isolates the *context* contribution; 1→3 is the total effect.
Generator = OpenAI `gpt-4o-mini`; embedder = `Octen/Octen-Embedding-0.6B` (fixed).

## 1. Config & setup

In [2]:
import os, re, json, time, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import pytrec_eval
from scipy import stats
from dotenv import load_dotenv
from openai import OpenAI

MODEL_NAME   = "Octen/Octen-Embedding-0.6B"
GEN_MODEL    = "gpt-4o-mini"
DATA_DIR     = Path("../nfcorpus")
SPLIT        = "test"
BASELINE_DIR = Path("results/baseline")
OUT_DIR      = Path("results/contextual")
TOP_K        = 1000
BATCH_SIZE   = 32
SEED         = 42

# ---- knobs ---------------------------------------------------------------
CHUNK_WORDS  = 80           # target chunk size (words); abstracts are short
AGG          = "max"        # pool chunk scores to a doc: 'max' or 'mean'
GEN_WORKERS  = 4            # keep low to stay under your OpenAI tokens-per-minute limit

OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
load_dotenv()
# max_retries lets the SDK auto-backoff on 429 rate limits (honors Retry-After)
client = OpenAI(max_retries=8)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — create a .env file with it."
print("device:", DEVICE, "| generator:", GEN_MODEL, "| embedder:", MODEL_NAME)

/Users/katerina_dimitrova/Downloads/Iris.ai RAG research/HyDE context Enrichment test/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps | generator: gpt-4o-mini | embedder: Octen/Octen-Embedding-0.6B


## 2. Load corpus, queries, qrels

In [4]:
def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def load_qrels(p):
    q = {}
    with open(p, encoding="utf-8") as f:
        next(f)
        for line in f:
            qid, did, s = line.rstrip("\n").split("\t")
            if int(s) > 0:
                q.setdefault(qid, {})[did] = int(s)
    return q

corpus = {r["_id"]: ((r.get("title") or ""), (r.get("text") or ""))
          for r in load_jsonl(DATA_DIR / "corpus.jsonl")}
corpus_ids = list(corpus.keys())
doc_index  = {d: i for i, d in enumerate(corpus_ids)}
qrels = load_qrels(DATA_DIR / "qrels" / f"{SPLIT}.tsv")
all_q = {r["_id"]: r["text"] for r in load_jsonl(DATA_DIR / "queries.jsonl")}
queries = {q: all_q[q] for q in qrels if q in all_q}
query_ids = list(queries.keys())
print(f"corpus {len(corpus)} docs | queries {len(queries)}")

corpus 3633 docs | queries 323


## 3. Chunk each abstract

Sentence-aware packing up to ~`CHUNK_WORDS` words. Short docs stay a single chunk.

In [5]:
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def chunk_doc(title, text, max_words=CHUNK_WORDS):
    sents = split_sentences(text) or [text]
    chunks, cur, n = [], [], 0
    for s in sents:
        w = len(s.split())
        if cur and n + w > max_words:
            chunks.append(" ".join(cur)); cur, n = [], 0
        cur.append(s); n += w
    if cur:
        chunks.append(" ".join(cur))
    # title prefixed to the first chunk so it isn't lost
    if title:
        chunks[0] = f"{title}. {chunks[0]}"
    return chunks

# flat chunk tables: parallel arrays
chunk_doc_id, chunk_text, full_doc_text = [], [], {}
for d in corpus_ids:
    title, text = corpus[d]
    full_doc_text[d] = (title + "\n" + text).strip()
    for c in chunk_doc(title, text):
        chunk_doc_id.append(d); chunk_text.append(c)
owner_idx = np.array([doc_index[d] for d in chunk_doc_id])
n_chunks = len(chunk_text)
per_doc = np.bincount(owner_idx)
print(f"{n_chunks} chunks from {len(corpus)} docs | chunks/doc: "
      f"min={per_doc.min()} mean={per_doc.mean():.2f} max={per_doc.max()}")

13210 chunks from 3633 docs | chunks/doc: min=1 mean=3.64 max=23


## 4. Generate situating context per chunk (cached + parallel)

Anthropic's prompt: give the LLM the **whole document** + the chunk, ask for a 1–2 sentence context.
Cached to `results/contextual/contexts.json` keyed by `docid::chunk_index`. Resumable.

In [6]:
CTX_PROMPT = (
    "<document>\n{doc}\n</document>\n\n"
    "Here is a chunk we want to situate within the whole document:\n"
    "<chunk>\n{chunk}\n</chunk>\n\n"
    "Give a short, succinct context (1-2 sentences) to situate this chunk within the overall "
    "document for the purpose of improving search retrieval of the chunk. Answer ONLY with the "
    "succinct context and nothing else."
)

# stable key per chunk = docid::position-within-doc
pos_counter = {}
chunk_keys = []
for d in chunk_doc_id:
    i = pos_counter.get(d, 0); pos_counter[d] = i + 1
    chunk_keys.append(f"{d}::{i}")

def gen_context(doc, chunk):
    # client(max_retries=8) auto-handles 429/timeout with exponential backoff
    msg = [{"role": "user", "content": CTX_PROMPT.format(doc=doc[:4000], chunk=chunk)}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msg,
                                       temperature=0.3, max_tokens=120, timeout=60)
    return r.choices[0].message.content.strip()

CPATH = OUT_DIR / "contexts.json"
contexts = json.loads(CPATH.read_text()) if CPATH.exists() else {}
todo = [i for i, k in enumerate(chunk_keys) if k not in contexts]
print(f"to generate: {len(todo)} contexts  (cached: {len(chunk_keys) - len(todo)})")

# Crash-safe: a failed call is recorded and skipped (NOT fatal); progress saved often.
# Re-run this cell to retry whatever failed/remains — it only touches missing chunks.
failed = []
lock = threading.Lock(); done = 0
try:
    with ThreadPoolExecutor(max_workers=GEN_WORKERS) as ex:
        futs = {ex.submit(gen_context, full_doc_text[chunk_doc_id[i]], chunk_text[i]): i for i in todo}
        for fut in tqdm(as_completed(futs), total=len(futs)):
            i = futs[fut]
            try:
                res = fut.result()
            except Exception as e:
                failed.append((i, type(e).__name__))
                continue
            with lock:
                contexts[chunk_keys[i]] = res; done += 1
                if done % 100 == 0:
                    CPATH.write_text(json.dumps(contexts))
finally:
    CPATH.write_text(json.dumps(contexts))   # always persist, even on Ctrl-C / interrupt

print(f"done. generated {done}, failed {len(failed)}, total cached {len(contexts)}/{len(chunk_keys)}.")
if failed:
    print("  failure types:", {t for _, t in failed}, "-> just re-run this cell to retry them.")
print("example ctx:", contexts.get(chunk_keys[0], "<none>")[:160])

to generate: 12310 contexts  (cached: 900)


  0%|          | 8/12310 [00:23<10:06:31,  2.96s/it]


KeyboardInterrupt: 

## 5. Embed chunks — plain vs contextualized (document space)

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 256

def embed(texts, prompt_name):
    return model.encode(texts, prompt_name=prompt_name, batch_size=BATCH_SIZE,
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=True).astype(np.float32)

# tolerate any still-missing contexts: fall back to the plain chunk (no KeyError)
missing = sum(1 for k in chunk_keys if k not in contexts)
if missing:
    print(f"NOTE: {missing} chunks have no context yet -> using plain chunk for those. "
          f"Re-run the generation cell to fill them in.")
ctx_texts = [(contexts.get(chunk_keys[i], "") + "\n" + chunk_text[i]).strip() for i in range(n_chunks)]

print("embedding plain chunks...")
plain_emb = embed(chunk_text, "document")
print("embedding contextualized chunks...")
ctx_emb = embed(ctx_texts, "document")
q_emb = embed([queries[q] for q in query_ids], "query")
print("chunk keys:", plain_emb.shape[0])

## 6. Retrieve (pool chunks → docs) + evaluate all conditions

In [ ]:
D = len(corpus_ids)
measures = {"ndcg_cut.1,3,5,10", "recall.3,5,10,20,100", "map", "P.10", "recip_rank", "success.1,5,10"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels, measures)

def pool_and_score(key_emb, owner):
    sims = q_emb @ key_emb.T                                   # (Q, n_keys)
    Q = sims.shape[0]
    if AGG == "max":
        scores = np.full((Q, D), -np.inf, dtype=np.float32)
        rows = np.broadcast_to(np.arange(Q)[:, None], sims.shape)
        cols = np.broadcast_to(owner[None, :], sims.shape)
        np.maximum.at(scores, (rows, cols), sims)
    else:
        scores = np.zeros((Q, D), dtype=np.float32); cnt = np.bincount(owner, minlength=D)
        for j, d in enumerate(owner):
            scores[:, d] += sims[:, j]
        scores /= np.maximum(cnt, 1)[None, :]
    k = min(TOP_K, D)
    topk = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    run = {}
    for i, qid in enumerate(query_ids):
        idx = topk[i]; order = idx[np.argsort(-scores[i, idx])]
        run[qid] = {corpus_ids[j]: float(scores[i, j]) for j in order}
    pq = evaluator.evaluate(run)
    avg = lambda m: float(np.mean([pq[q][m] for q in pq]))
    metrics = {
        "NDCG@1": avg("ndcg_cut_1"), "NDCG@3": avg("ndcg_cut_3"), "NDCG@5": avg("ndcg_cut_5"), "NDCG@10": avg("ndcg_cut_10"),
        "Recall@3": avg("recall_3"), "Recall@5": avg("recall_5"), "Recall@10": avg("recall_10"),
        "Recall@20": avg("recall_20"), "Recall@100": avg("recall_100"),
        "Hit@1": avg("success_1"), "Hit@5": avg("success_5"), "Hit@10": avg("success_10"),
        "MAP": avg("map"), "P@10": avg("P_10"), "MRR": avg("recip_rank"),
    }
    return metrics, {q: pq[q]["ndcg_cut_10"] for q in pq}, run

chunks_metrics, chunks_pq, _        = pool_and_score(plain_emb, owner_idx)
ctx_metrics,    ctx_pq,    ctx_run  = pool_and_score(ctx_emb,   owner_idx)
print("scored chunks-only and contextual.")

## 7. Save + three-way comparison (baseline / chunks-only / contextual)

In [ ]:
# save the contextual run as the headline method output
with open(OUT_DIR / "run.tsv", "w") as f:
    for qid in query_ids:
        for rank, (did, sc) in enumerate(ctx_run[qid].items(), start=1):
            f.write(f"{qid}\tQ0\t{did}\t{rank}\t{sc:.6f}\tcontextual\n")
(OUT_DIR / "metrics.json").write_text(json.dumps(ctx_metrics, indent=2))
(OUT_DIR / "metrics_chunks_only.json").write_text(json.dumps(chunks_metrics, indent=2))
(OUT_DIR / "per_query_ndcg10.json").write_text(json.dumps(ctx_pq, indent=2))
(OUT_DIR / "config.json").write_text(json.dumps({
    "enrichment": "Contextual Retrieval (chunk-then-contextualize)", "model_name": MODEL_NAME,
    "generator": GEN_MODEL, "chunk_words": CHUNK_WORDS, "agg": AGG, "n_chunks": int(n_chunks),
    "dataset": "NFCorpus", "split": SPLIT, "n_queries": len(queries), "seed": SEED,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}, indent=2))

base_metrics = json.loads((BASELINE_DIR / "metrics.json").read_text())
base_pq = json.loads((BASELINE_DIR / "per_query_ndcg10.json").read_text())

cols = [("baseline", base_metrics), ("chunks-only", chunks_metrics), ("contextual", ctx_metrics)]
print(f"{'metric':<12}" + "".join(f"{n:>13}" for n, _ in cols))
print("-" * 51)
for m in base_metrics:
    print(f"{m:<12}" + "".join(f"{mt.get(m, float('nan')):>13.4f}" for _, mt in cols))

def compare(name, pq):
    qs = [q for q in query_ids if q in base_pq]
    d = np.array([pq[q] - base_pq[q] for q in qs])
    w, l = int((d > 1e-9).sum()), int((d < -1e-9).sum())
    nz = d[np.abs(d) > 1e-9]; p = stats.wilcoxon(nz).pvalue if len(nz) else 1.0
    sig = "SIGNIFICANT" if p < 0.05 else "ns"
    print(f"  {name:<12} dNDCG@10={d.mean():+.4f}  wins/losses={w}/{l}  Wilcoxon p={p:.3g} ({sig})")

print("\nvs baseline (per-query NDCG@10):")
compare("chunks-only", chunks_pq)
compare("contextual", ctx_pq)
# isolate the context effect: contextual vs chunks-only
qs = list(ctx_pq)
dctx = np.array([ctx_pq[q] - chunks_pq[q] for q in qs])
nz = dctx[np.abs(dctx) > 1e-9]; pc = stats.wilcoxon(nz).pvalue if len(nz) else 1.0
print(f"\ncontext effect (contextual - chunks-only): dNDCG@10={dctx.mean():+.4f}  "
      f"Wilcoxon p={pc:.3g} ({'SIGNIFICANT' if pc<0.05 else 'ns'})")